# PINN v7 - integrator comparison (Euler / closed-form / RK4)

Stripped-down v6: **same** synthetic data, **same** Stage C noise, the **same** bias-supervised trainer, and the **same** high-res evaluation that produced Euler 14.6 m and closed-form 14.3 m. The only change is **RK4** added as a third integrator, in the two places it lives: the data generator's ground-truth integration and the model's forward step. Trains all three under identical settings in one run so the comparison is clean, then prints the drift table.

Set CONFIG to the real values and run top to bottom on the GPU box.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## Config

In [ ]:
# --- CONFIG (edit for the real run) ---
N_TRAIN  = 500      # windows per integrator
N_VAL    = 100
N_EPOCHS = 600
N_SEEDS  = 5       # noise realizations averaged per eval window
LAMBDA_BIAS = 100_000.0
INTEGRATORS = ['euler', 'closed_form', 'rk4']
print('config:', dict(N_TRAIN=N_TRAIN, N_VAL=N_VAL, N_EPOCHS=N_EPOCHS, N_SEEDS=N_SEEDS))

## 1. Synthetic data
`integrator` sets how the ground-truth trajectory is built (Euler / closed-form / RK4) from the same IMU.

In [ ]:
def _prim_straight(n_steps):
    return np.zeros(n_steps), np.zeros(n_steps)

def _prim_arc(n_steps, v_in, radius, direction):
    # Constant-radius turn at the entering speed; ax=0 keeps v constant.
    wz_val = direction * v_in / radius
    return np.zeros(n_steps), np.full(n_steps, wz_val)

def _prim_speed_change(n_steps, target_ax):
    return np.full(n_steps, target_ax), np.zeros(n_steps)


def compose_window(primitives_spec, v0, yaw0, dt=0.1, duration=30.0, integrator='euler'):
    """integrator: 'euler' or 'closed_form'. Both produce same IMU, different x_gt/y_gt."""
    N = int(round(duration / dt)) + 1
    ax_arr = np.zeros(N)
    wz_arr = np.zeros(N)

    n_steps_per = [int(round(p[2] / dt)) for p in primitives_spec]
    n_steps_per[-1] += (N - 1) - sum(n_steps_per)

    v_now = v0
    idx = 0
    for (ptype, kwargs, _), n_steps in zip(primitives_spec, n_steps_per):
        if ptype == 'straight':
            ax_seg, wz_seg = _prim_straight(n_steps)
        elif ptype == 'arc':
            ax_seg, wz_seg = _prim_arc(n_steps, v_now, kwargs['radius'], kwargs['direction'])
        elif ptype == 'speed_change':
            ax_seg, wz_seg = _prim_speed_change(n_steps, kwargs['ax'])
            v_now = v_now + kwargs['ax'] * (n_steps * dt)
        else:
            raise ValueError(ptype)
        ax_arr[idx:idx + n_steps] = ax_seg
        wz_arr[idx:idx + n_steps] = wz_seg
        idx += n_steps
    ax_arr[-1] = ax_arr[-2]; wz_arr[-1] = wz_arr[-2]   # avoid trailing-zero cosmetic

    # Integrate forward using the requested scheme
    x = np.zeros(N); y = np.zeros(N); v = np.zeros(N); yaw = np.zeros(N)
    x[0], y[0], v[0], yaw[0] = 0.0, 0.0, v0, yaw0
    eps = 1e-6
    for i in range(N - 1):
        a, w   = ax_arr[i], wz_arr[i]
        v_i, psi = v[i], yaw[i]
        if integrator == 'euler':
            x[i+1] = x[i] + v_i * np.cos(psi) * dt
            y[i+1] = y[i] + v_i * np.sin(psi) * dt
        elif integrator == 'closed_form':
            if abs(w) < eps:
                x[i+1] = x[i] + v_i * np.cos(psi) * dt
                y[i+1] = y[i] + v_i * np.sin(psi) * dt
            else:
                x[i+1] = x[i] + (v_i / w) * (np.sin(psi + w*dt) - np.sin(psi))
                y[i+1] = y[i] - (v_i / w) * (np.cos(psi + w*dt) - np.cos(psi))
        elif integrator == 'rk4':
            def _d(xx, yy, vv, pp):
                return vv*np.cos(pp), vv*np.sin(pp), a, w
            k1 = _d(x[i], y[i], v_i, psi)
            k2 = _d(x[i]+0.5*dt*k1[0], y[i]+0.5*dt*k1[1], v_i+0.5*dt*k1[2], psi+0.5*dt*k1[3])
            k3 = _d(x[i]+0.5*dt*k2[0], y[i]+0.5*dt*k2[1], v_i+0.5*dt*k2[2], psi+0.5*dt*k2[3])
            k4 = _d(x[i]+dt*k3[0],     y[i]+dt*k3[1],     v_i+dt*k3[2],     psi+dt*k3[3])
            x[i+1] = x[i] + dt/6.0*(k1[0]+2*k2[0]+2*k3[0]+k4[0])
            y[i+1] = y[i] + dt/6.0*(k1[1]+2*k2[1]+2*k3[1]+k4[1])
        else:
            raise ValueError(integrator)
        yaw[i+1] = psi + w * dt
        v[i+1]   = v_i + a * dt
    ay_arr = v * wz_arr
    t = np.arange(0, duration + dt/2, dt)

    return {
        't': t, 'dt': np.full(N, dt),
        'x_gt': x, 'y_gt': y, 'v_gt': v, 'yaw_gt': yaw,
        'ax': ax_arr, 'ay': ay_arr, 'wz': wz_arr,
        'v0': float(v0), 'yaw0': float(yaw0),
        'drive': 'synthetic_composed',
        'gt_integrator': integrator,
    }


def sample_complexity_window(complexity, seed, integrator='euler'):
    """Same as before but propagates integrator to compose_window."""
    rng = np.random.default_rng(seed)
    v0   = float(rng.uniform(5.0, 20.0))
    yaw0 = float(rng.uniform(-np.pi, np.pi))
    if complexity == 'simple':
        n_prims      = int(rng.choice([1, 2]))
        prim_probs   = {'straight': 0.60, 'arc': 0.20, 'speed_change': 0.20}
        radius_range = (60.0, 150.0); ax_range = (-0.3, 0.3)
    else:
        n_prims      = int(rng.choice([3, 4, 5]))
        prim_probs   = {'straight': 0.30, 'arc': 0.40, 'speed_change': 0.30}
        radius_range = (20.0, 100.0); ax_range = (-1.0, 1.0)
    names = list(prim_probs.keys())
    probs = np.array(list(prim_probs.values())); probs /= probs.sum()
    raw = rng.uniform(0.7, 1.3, n_prims); raw *= 30.0 / raw.sum()
    durations = raw.tolist()
    while min(durations) < 3.0 and len(durations) > 1:
        i = int(np.argmin(durations))
        j = i + 1 if i + 1 < len(durations) else i - 1
        durations[j] += durations[i]; durations.pop(i)
    s = sum(durations); durations = [d * 30.0 / s for d in durations]
    spec = []
    for d in durations:
        ptype = str(rng.choice(names, p=probs))
        if ptype == 'arc':
            kw = {'radius': float(rng.uniform(*radius_range)),
                  'direction': float(rng.choice([-1.0, 1.0]))}
        elif ptype == 'speed_change':
            kw = {'ax': float(rng.uniform(*ax_range))}
        else:
            kw = {}
        spec.append((ptype, kw, d))
    w = compose_window(spec, v0=v0, yaw0=yaw0, integrator=integrator)
    w['spec']       = spec
    w['complexity'] = complexity
    return w, spec


def make_dataset_v2(n_windows, seed_base=0, p_simple=0.8, integrator='euler'):
    rng = np.random.default_rng(seed_base)
    windows = []
    for i in range(n_windows):
        complexity = 'simple' if rng.random() < p_simple else 'complex'
        w, spec = sample_complexity_window(complexity, seed=seed_base + i, integrator=integrator)
        windows.append(w)
    n_simp = sum(1 for w in windows if w['complexity'] == 'simple')
    print(f"make_dataset_v2(n={n_windows}, integrator={integrator!r}):  "
          f"simple={n_simp}, complex={len(windows)-n_simp}")
    return windows

## 2. Stage C noise + batch stacker
Constant bias + slow drift + white noise; the stacker also returns the true per-timestep bias profile used for supervision.

In [ ]:
NOISE_PARAMS_C = dict(
    sigma_b0_a=0.10,
    sigma_b0_w=0.015,
    sigma_a=0.05,          # white noise on accel (m/s²)
    sigma_w=0.01,          # white noise on gyro  (rad/s)
    sigma_brwa=0.001,      # accel bias random walk (m/s²/√s)
    sigma_brww=0.0002,     # gyro bias random walk  (rad/s/√s)
)

def inject_full_noise(window, noise_params, seed):
    """Constant bias + white noise + bias random walk."""
    rng = np.random.default_rng(seed)
    N = len(window['t'])
    dt = float(window['dt'][0])
    b0_ax = rng.normal(0.0, noise_params['sigma_b0_a'])
    b0_wz = rng.normal(0.0, noise_params['sigma_b0_w'])
    white_ax = rng.normal(0.0, noise_params['sigma_a'], N)
    white_wz = rng.normal(0.0, noise_params['sigma_w'], N)
    brw_ax = np.cumsum(rng.normal(0.0, noise_params['sigma_brwa']*np.sqrt(dt), N))
    brw_wz = np.cumsum(rng.normal(0.0, noise_params['sigma_brww']*np.sqrt(dt), N))
    bias_a_profile = b0_ax + brw_ax
    bias_w_profile = b0_wz + brw_wz
    return {
        'ax': window['ax'] + bias_a_profile + white_ax,
        'ay': window['ay'],
        'wz': window['wz'] + bias_w_profile + white_wz,
        'bias_a_profile': bias_a_profile,
        'bias_w_profile': bias_w_profile,
    }


def stack_batch_full_noise_v2(windows, noise_params, base_seed):
    """Same as stack_batch_full_noise but also returns per-timestep bias profiles."""
    noisy = [inject_full_noise(w, noise_params, base_seed + i) for i, w in enumerate(windows)]
    ax = torch.tensor(np.stack([n['ax'] for n in noisy]), dtype=torch.float32, device=DEVICE)
    ay = torch.tensor(np.stack([n['ay'] for n in noisy]), dtype=torch.float32, device=DEVICE)
    wz = torch.tensor(np.stack([n['wz'] for n in noisy]), dtype=torch.float32, device=DEVICE)
    dt = torch.tensor(np.stack([w['dt'] for w in windows]), dtype=torch.float32, device=DEVICE)
    v0 = torch.tensor([w['v0']   for w in windows], dtype=torch.float32, device=DEVICE)
    yaw0 = torch.tensor([w['yaw0'] for w in windows], dtype=torch.float32, device=DEVICE)
    b_a_mean = torch.tensor([n['bias_a_profile'].mean() for n in noisy],
                            dtype=torch.float32, device=DEVICE)
    b_w_mean = torch.tensor([n['bias_w_profile'].mean() for n in noisy],
                            dtype=torch.float32, device=DEVICE)
    # NEW: per-timestep bias profile tensors (B, T)
    ba_profile = torch.tensor(np.stack([n['bias_a_profile'] for n in noisy]),
                              dtype=torch.float32, device=DEVICE)
    bw_profile = torch.tensor(np.stack([n['bias_w_profile'] for n in noisy]),
                              dtype=torch.float32, device=DEVICE)
    return ax, ay, wz, dt, v0, yaw0, b_a_mean, b_w_mean, ba_profile, bw_profile

## 3. Model
GRU bias estimator + hard-coded integrator. RK4 sits beside Euler and closed-form.

In [ ]:
class BiasGRU(nn.Module):
    def __init__(self, hidden=32, n_layers=1):
        super().__init__()
        self.gru  = nn.GRU(input_size=6, hidden_size=hidden,
                           num_layers=n_layers, batch_first=True)
        self.head = nn.Linear(hidden, 2)
        # Small (NOT zero) weights so initial bias predictions are tiny
        # but gradients can flow back into the GRU.
        nn.init.normal_(self.head.weight, std=1e-3)
        nn.init.zeros_(self.head.bias)

    def forward(self, imu_seq, state_seq):
        x = torch.cat([imu_seq, state_seq], dim=-1)
        h, _ = self.gru(x)
        bias = self.head(h)
        return bias


class GreyBoxPINN(nn.Module):
    def __init__(self, hidden=32, integrator='euler'):
        super().__init__()
        self.bias_net = BiasGRU(hidden=hidden)
        self.integrator = integrator

    def forward(self, ax, ay, wz, dt, v0, yaw0):
        B, T = ax.shape
        v0_seq     = v0.unsqueeze(1).expand(B, T)
        sin_yaw0_s = torch.sin(yaw0).unsqueeze(1).expand(B, T)
        cos_yaw0_s = torch.cos(yaw0).unsqueeze(1).expand(B, T)
        imu_seq    = torch.stack([ax, ay, wz], dim=-1)
        state_seq  = torch.stack([v0_seq, sin_yaw0_s, cos_yaw0_s], dim=-1)
        bias_seq   = self.bias_net(imu_seq, state_seq)

        x_list   = [torch.zeros(B, device=ax.device)]
        y_list   = [torch.zeros(B, device=ax.device)]
        v_list   = [v0]
        yaw_list = [yaw0]
        eps = 1e-4

        for i in range(T - 1):
            b_a = bias_seq[:, i, 0]; b_w = bias_seq[:, i, 1]
            ax_clean = ax[:, i] - b_a
            wz_clean = wz[:, i] - b_w
            dt_i  = dt[:, i]
            v_cur = v_list[-1]; psi = yaw_list[-1]

            if self.integrator == 'euler':
                x_next = x_list[-1] + v_cur * torch.cos(psi) * dt_i
                y_next = y_list[-1] + v_cur * torch.sin(psi) * dt_i
            elif self.integrator == 'closed_form':
                psi_new = psi + wz_clean * dt_i
                mask    = torch.abs(wz_clean) < eps
                safe_w  = torch.where(mask, torch.full_like(wz_clean, eps), wz_clean)
                x_cf = x_list[-1] + v_cur * (torch.sin(psi_new) - torch.sin(psi)) / safe_w
                y_cf = y_list[-1] - v_cur * (torch.cos(psi_new) - torch.cos(psi)) / safe_w
                x_st = x_list[-1] + v_cur * torch.cos(psi) * dt_i
                y_st = y_list[-1] + v_cur * torch.sin(psi) * dt_i
                x_next = torch.where(mask, x_st, x_cf)
                y_next = torch.where(mask, y_st, y_cf)
            elif self.integrator == 'rk4':
                a = ax_clean; w = wz_clean
                def _d(xx, yy, vv, pp):
                    return vv*torch.cos(pp), vv*torch.sin(pp), a, w
                k1 = _d(x_list[-1], y_list[-1], v_cur, psi)
                k2 = _d(x_list[-1]+0.5*dt_i*k1[0], y_list[-1]+0.5*dt_i*k1[1], v_cur+0.5*dt_i*k1[2], psi+0.5*dt_i*k1[3])
                k3 = _d(x_list[-1]+0.5*dt_i*k2[0], y_list[-1]+0.5*dt_i*k2[1], v_cur+0.5*dt_i*k2[2], psi+0.5*dt_i*k2[3])
                k4 = _d(x_list[-1]+dt_i*k3[0],     y_list[-1]+dt_i*k3[1],     v_cur+dt_i*k3[2],     psi+dt_i*k3[3])
                x_next = x_list[-1] + dt_i/6.0*(k1[0]+2*k2[0]+2*k3[0]+k4[0])
                y_next = y_list[-1] + dt_i/6.0*(k1[1]+2*k2[1]+2*k3[1]+k4[1])
            else:
                raise ValueError(self.integrator)

            v_next   = v_cur + ax_clean * dt_i
            yaw_next = psi   + wz_clean * dt_i
            x_list.append(x_next); y_list.append(y_next)
            v_list.append(v_next); yaw_list.append(yaw_next)

        x   = torch.stack(x_list,   dim=1)
        y   = torch.stack(y_list,   dim=1)
        v   = torch.stack(v_list,   dim=1)
        yaw = torch.stack(yaw_list, dim=1)
        return x, y, v, yaw, bias_seq

## 4. Trainer
position MSE + smoothness + lambda_bias * (bias MSE). Identical to v6.

In [ ]:
def train_noisy_with_bias(model, train_windows, val_windows, noise_params,
                          stack_fn=None,
                          n_epochs=600, batch_size=32, lr=1e-3, log_every=20,
                          lambda_smooth=100.0, lambda_bias=0.0):
    """
    Trainer with optional direct bias supervision.
    lambda_bias = 0 -> same behaviour as train_noisy.
    lambda_bias > 0 -> add MSE(predicted_bias_profile, true_bias_profile) to the loss.
    stack_fn must return 10 values (see stack_batch_full_noise_v2).
    """
    if stack_fn is None:
        stack_fn = stack_batch_full_noise_v2

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    history = {'epoch': [], 'pos_loss': [], 'smooth_loss': [], 'bias_loss': [],
               'val_drift_30s': [], 'bias_corr_a': [], 'bias_corr_w': []}

    for ep in range(n_epochs):
        model.train()
        idxs = np.random.permutation(len(train_windows))
        epoch_seed = 1_000_000 + ep * 10_000
        last_pos, last_smooth, last_bias = 0.0, 0.0, 0.0

        for start in range(0, len(train_windows), batch_size):
            batch_idx = idxs[start:start + batch_size]
            batch = [train_windows[i] for i in batch_idx]
            ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b, _, _, ba_prof, bw_prof = stack_fn(
                batch, noise_params, epoch_seed + start)
            x_gt = torch.tensor(np.stack([w['x_gt'] for w in batch]),
                                dtype=torch.float32, device=DEVICE)
            y_gt = torch.tensor(np.stack([w['y_gt'] for w in batch]),
                                dtype=torch.float32, device=DEVICE)

            x_p, y_p, _, _, bias_seq = model(ax_b, ay_b, wz_b, dt_b, v0_b, yaw0_b)

            pos_loss    = ((x_p - x_gt)**2 + (y_p - y_gt)**2).mean()
            smooth_loss = ((bias_seq[:, 1:, :] - bias_seq[:, :-1, :])**2).mean()
            bias_loss   = ((bias_seq[:, :, 0] - ba_prof)**2 +
                           (bias_seq[:, :, 1] - bw_prof)**2).mean()

            loss = pos_loss + lambda_smooth * smooth_loss + lambda_bias * bias_loss

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            last_pos    = pos_loss.item()
            last_smooth = smooth_loss.item()
            last_bias   = bias_loss.item()
        scheduler.step()

        if ep % log_every == 0 or ep == n_epochs - 1:
            model.eval()
            with torch.no_grad():
                val_drifts, true_bas, pred_bas, true_bws, pred_bws = [], [], [], [], []
                for vw in val_windows:
                    for s in range(3):
                        ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v, b_a_t, b_w_t, _, _ = stack_fn(
                            [vw], noise_params, base_seed=900_000 + s * 100)
                        x_v, y_v, _, _, bias_v = model(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
                        drift = float(torch.sqrt(
                            (x_v[0, -1] - vw['x_gt'][-1])**2 +
                            (y_v[0, -1] - vw['y_gt'][-1])**2))
                        val_drifts.append(drift)
                        true_bas.append(float(b_a_t[0]))
                        pred_bas.append(float(bias_v[:,:,0].mean()))
                        true_bws.append(float(b_w_t[0]))
                        pred_bws.append(float(bias_v[:,:,1].mean()))
                corr_a = np.corrcoef(true_bas, pred_bas)[0, 1]
                corr_w = np.corrcoef(true_bws, pred_bws)[0, 1]

            history['epoch'].append(ep)
            history['pos_loss'].append(last_pos)
            history['smooth_loss'].append(last_smooth)
            history['bias_loss'].append(last_bias)
            history['val_drift_30s'].append(float(np.mean(val_drifts)))
            history['bias_corr_a'].append(float(corr_a))
            history['bias_corr_w'].append(float(corr_w))
            print(f"  ep {ep:4d}  pos={last_pos:7.2f}  bias={last_bias:.5f}  "
                  f"val_drift={np.mean(val_drifts):6.2f}m  "
                  f"corr(b_a)={corr_a:+.3f}  corr(b_w)={corr_w:+.3f}  "
                  f"lr={opt.param_groups[0]['lr']:.5f}")
    return history

## 5. Evaluation
Endpoint drift vs the 100x high-res reference - the fair metric.

In [ ]:
def compute_highres_gt(window, sub=100):
    """Returns highres x, y by Euler-integrating the SAME IMU at 100x finer dt."""
    N = len(window['t'])
    dt_orig = float(window['dt'][0])
    dt_fine = dt_orig / sub
    ax_fine = np.repeat(window['ax'][:-1], sub)
    wz_fine = np.repeat(window['wz'][:-1], sub)
    N_fine = len(ax_fine) + 1
    x = np.zeros(N_fine); y = np.zeros(N_fine)
    v = np.zeros(N_fine); yaw = np.zeros(N_fine)
    v[0], yaw[0] = window['v0'], window['yaw0']
    for i in range(N_fine - 1):
        x[i+1]   = x[i]   + v[i] * np.cos(yaw[i]) * dt_fine
        y[i+1]   = y[i]   + v[i] * np.sin(yaw[i]) * dt_fine
        v[i+1]   = v[i]   + ax_fine[i] * dt_fine
        yaw[i+1] = yaw[i] + wz_fine[i] * dt_fine
    idxs = np.arange(N) * sub
    return x[idxs], y[idxs]


def evaluate_drift(model, test_windows, noise_params, n_seeds=5, seed_base=12_000_000):
    # Mean endpoint drift (m) vs the 100x high-res reference, over windows x noise seeds.
    # Same fair metric that produced Euler 14.6 / closed-form 14.3.
    model.eval()
    drifts = []
    with torch.no_grad():
        for vw in test_windows:
            x_hr, y_hr = compute_highres_gt(vw, sub=100)
            for s in range(n_seeds):
                noisy = inject_full_noise(vw, noise_params, seed=seed_base + s)
                ax_v = torch.tensor(noisy['ax'][None], dtype=torch.float32, device=DEVICE)
                ay_v = torch.tensor(noisy['ay'][None], dtype=torch.float32, device=DEVICE)
                wz_v = torch.tensor(noisy['wz'][None], dtype=torch.float32, device=DEVICE)
                dt_v = torch.tensor(vw['dt'][None],    dtype=torch.float32, device=DEVICE)
                v0_v = torch.tensor([vw['v0']],        dtype=torch.float32, device=DEVICE)
                yaw0_v = torch.tensor([vw['yaw0']],    dtype=torch.float32, device=DEVICE)
                x_p, y_p, _, _, _ = model(ax_v, ay_v, wz_v, dt_v, v0_v, yaw0_v)
                d = float(np.sqrt((float(x_p[0, -1]) - x_hr[-1])**2 +
                                  (float(y_p[0, -1]) - y_hr[-1])**2))
                drifts.append(d)
    return float(np.mean(drifts)), float(np.std(drifts))

## 6. Run
Train all three integrators under identical settings and compare. On an RTX 3060 this is three short trainings; euler/closed-form should land near 14.6/14.3, confirming the harness matches v6.

In [ ]:
# Fixed test set (seeds far outside train/val); IMU + high-res GT are integrator-independent.
test_windows = ([sample_complexity_window('simple',  seed=900_000 + i)[0] for i in range(5)] +
                [sample_complexity_window('complex', seed=910_000 + i)[0] for i in range(5)])

results = {}
for integ in INTEGRATORS:
    print(f"\n{'='*56}")
    print(f"  Training integrator = {integ}")
    print('='*56)
    train_w = make_dataset_v2(N_TRAIN, seed_base=200_000, integrator=integ)
    val_w   = make_dataset_v2(N_VAL,   seed_base=210_000, integrator=integ)
    torch.manual_seed(0); np.random.seed(0)
    model = GreyBoxPINN(integrator=integ).to(DEVICE)
    train_noisy_with_bias(model, train_w, val_w, NOISE_PARAMS_C,
                          n_epochs=N_EPOCHS, lambda_bias=LAMBDA_BIAS)
    mean_d, std_d = evaluate_drift(model, test_windows, NOISE_PARAMS_C, n_seeds=N_SEEDS)
    results[integ] = mean_d
    torch.save(model.state_dict(), f"pinn_n{N_TRAIN}_{integ}.pt")
    print(f"  >>> {integ}: mean endpoint drift vs high-res = {mean_d:.2f} m (+/- {std_d:.2f})")

print('='*44)
print('   INTEGRATOR COMPARISON  (lower = better)')
print('='*44)
for integ, d in sorted(results.items(), key=lambda kv: kv[1]):
    print(f"   {integ:>12} : {d:6.2f} m")
winner = min(results, key=results.get)
print(f"   Winner: {winner}  ({results[winner]:.2f} m)")
print('   (expected ballpark from v6: euler ~14.6, closed_form ~14.3)')